# Exercice 2.2 : un outil réseau instable (40 min)

> 🗓️ **Session en 4 jours :** démo commentée par le formateur pendant `J2_04`. À refaire en autonomie si vous le souhaitez ; le corrigé est distribué.

> Prérequis : J2-04. Notebook **autonome**, asynchrone (`await` directement dans les cellules).

**Contexte :** `ping_serveur` interroge la supervision de NordTech, qui tombe souvent. Aujourd'hui, une panne de supervision fait **planter** l'agent support. Votre mission : le rendre tolérant aux pannes, sans jamais inventer un résultat, et le **prouver** dans Langfuse.

| Étape | Consigne |
|---|---|
| 0. Constat | exécutez les 4 scénarios **sans protection** et notez ce qui casse |
| 1. Retry | `RetryPolicy` sur le nœud `tools` : 3 tentatives, `retry_on=ConnectionError`, attente initiale 0,2 s |
| 2. Timeout outils | `TimeoutPolicy(run_timeout=1)` sur `tools` (le scénario « lente » répond en 3 s) |
| 3. Repli outils | `error_handler` sur `tools` : un `ToolMessage` d'erreur **par appel d'outil en attente**, puis `goto="agent"` |
| 4. Repli agent | timeout sur `agent` (réglé pour **votre** machine) et `error_handler` qui répond poliment puis termine |
| 5. Preuve | les 4 scénarios passent sans exception ; dans Langfuse (session `ex22-<initiales>`), retrouvez les tentatives en échec |

**Critères de réussite :** aucune exception ne remonte · en panne, la réponse **dit** que la supervision est indisponible (aucun résultat inventé) · le scénario « instable » réussit grâce aux retries · vous savez montrer, dans une trace, l'étape qui a été réessayée.

## 0. Mise en route et agent fourni

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.errors import NodeError
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, RetryPolicy, TimeoutPolicy

from helpdesk_ops.outils import chercher_kb, statut_serveur
from helpdesk_ops.reseau import RESEAU, ping_serveur

OUTILS = [statut_serveur, chercher_kb, ping_serveur]
llm_outils = llm.bind_tools(OUTILS)
SYSTEME = SystemMessage(
    "Tu es l'agent support N1 de NordTech SA. Réponds en français, en 2 phrases maximum. "
    "Pour savoir si un serveur répond, utilise ping_serveur. Si un outil est indisponible, dis-le : n'invente jamais son résultat."
)


async def agent(state: MessagesState) -> dict:
    return {"messages": [await llm_outils.ainvoke([SYSTEME, *state["messages"]])]}


def construire(options_agent: dict | None = None, options_tools: dict | None = None):
    """Fourni : l'agent support, async. Vous ne changez QUE les options passées aux deux nœuds."""
    b = StateGraph(MessagesState)
    b.add_node("agent", agent, **(options_agent or {}))
    b.add_node("tools", ToolNode(OUTILS), **(options_tools or {}))
    b.add_edge(START, "agent")
    b.add_conditional_edges("agent", tools_condition)
    b.add_edge("tools", "agent")
    return b.compile()


QUESTION = "Le serveur srv-erp-02 répond-il au ping ?"
SCENARIOS = {
    "stable": {},
    "instable": {"taux_echec": 0.5, "graine": 3},
    "panne": {"echecs_avant_succes": 99},
    "lente": {"latence_s": 3.0},
}


async def essayer(graphe, scenario: str, session: str) -> None:
    RESEAU.configurer(**SCENARIOS[scenario])
    try:
        with propagate_attributes(session_id=session, tags=["jour-2", "ex22", scenario]):
            etat = await graphe.ainvoke({"messages": [HumanMessage(QUESTION)]},
                                        config={"callbacks": [langfuse_handler], "run_name": f"ex22-{scenario}", "recursion_limit": 10})
        print(f"✅ {scenario:<9} | {RESEAU.appels} appel(s) réseau | {etat['messages'][-1].content[:110]}")
    except Exception as erreur:
        print(f"💥 {scenario:<9} | {RESEAU.appels} appel(s) réseau | {type(erreur).__name__} : {str(erreur)[:80]}")

## 1. Le constat, sans protection

In [ ]:
INITIALES = "xx"   # TODO
fragile = construire()
for scenario in SCENARIOS:
    await essayer(fragile, scenario, f"ex22-{INITIALES}")

## 2. Vos protections

In [ ]:
def outils_en_echec(state: MessagesState, error: NodeError) -> Command:
    ...  # TODO : un ToolMessage(status="error", tool_call_id=…, name=…) par appel en attente, puis goto="agent"


def agent_en_echec(state: MessagesState, error: NodeError) -> Command:
    ...  # TODO : un AIMessage poli, puis goto=END


OPTIONS_TOOLS = {}   # TODO : retry_policy, timeout, error_handler
OPTIONS_AGENT = {}   # TODO : timeout, error_handler

robuste = construire(OPTIONS_AGENT, OPTIONS_TOOLS)
for scenario in SCENARIOS:
    await essayer(robuste, scenario, f"ex22-{INITIALES}")
langfuse.flush()

## 3. Dans Langfuse

1. Scénario « instable » : combien d'observations `tools` ? Lesquelles sont en erreur (niveau `ERROR`) ?
2. Scénario « panne » : que reçoit le modèle ? Que répond-il ?
3. Quelle est la durée totale du scénario « lente » ? D'où vient-elle ?